# Feature Extraction for VSX → TESS Variable-Star Classification

This notebook extracts interpretable machine-learning features from standardized TESS light-curve FITS files.

It is designed for the current research pipeline:

- **Input**: metadata parquet containing `lightCurvePath`
- **Light curve source**: standardized FITS files already saved by `TessDataDownloader.py`
- **Output**: feature parquet, one row per star
- **Parallel processing**: configurable thread pool, default `WorkerCount = 16`

Important leakage rule:

> Feature extraction here is safe before train/validation/test split because every feature is computed independently per star.  
> Dataset-level scaling, PCA, feature selection, and model fitting must happen only after train/validation/test split.


# Complete Feature List

## 1. Identification / Metadata Features

| Feature | Meaning | Method / Source |
|---|---|---|
| `family` | VSX family label, e.g. `RRLYR`, `CEPHEID`, `ECLIPSING` | copied from metadata |
| `VSXType` | Original/mapped VSX type | copied from metadata |
| `VSXId` | VSX identifier | copied from metadata |
| `Name` | VSX object name | copied from metadata |
| `ticId` | TIC identifier, if present | copied from metadata |
| `bestTicId` | selected TIC match | copied from metadata |
| `ticDistanceArcmin` | angular distance to selected TIC match | copied from metadata |
| `lightCurvePath` | standardized FITS path | copied from metadata |
| `rawLightCurvePath` | raw FITS path | copied from metadata |
| `trendFlag` | trend flag from trend analysis, if present | copied from metadata |
| `trendScore` | trend score, if present | copied from metadata |
| `adfPValue` | stationarity metric, if present | copied from metadata |
| `QualityLabel` | light curve quality label | copied/inferred from metadata |
| `QualityScore` | numeric quality encoding | clean=0, acceptable=1, caution=2, poor=3 |
| `Provenance` | data source: SPOC / QLP / TESSCut | copied/inferred from metadata |
| `ProvenanceScore` | numeric provenance encoding | SPOC=0, QLP=1, TESSCut=2 |
| `OriginalFluxMedian` | original pre-standardization median flux | copied from metadata |
| `OriginalFluxStd` | original pre-standardization flux std | copied from metadata |
| `OriginalFluxSnr` | original flux signal-to-noise estimate | copied from metadata |
| `LowSnr` | low-SNR flag | copied from metadata |
| `LowQualityLightCurve` | low-quality light curve flag | copied from metadata |

## 2. Data Coverage / Cadence Features

| Feature | Meaning | Method |
|---|---|---|
| `CadenceCount` | number of finite flux observations | `len(Flux)` |
| `TimeSpanDays` | total time baseline | `max(Time) - min(Time)` |
| `MedianCadenceDays` | typical time spacing between observations | `median(diff(Time))` |

## 3. Flux Distribution Features

| Feature | Meaning | Method |
|---|---|---|
| `FluxMean` | average standardized brightness | `mean(Flux)` |
| `FluxStd` | standard deviation of brightness | `std(Flux)` |
| `FluxVariance` | variance of brightness | `var(Flux)` |
| `FluxMedian` | median brightness | `median(Flux)` |
| `FluxMad` | robust variability measure | `median(abs(Flux - median))` |
| `FluxMin` | minimum flux | `min(Flux)` |
| `FluxMax` | maximum flux | `max(Flux)` |
| `FluxP01` | 1st percentile | `percentile(Flux, 1)` |
| `FluxP05` | 5th percentile | `percentile(Flux, 5)` |
| `FluxP10` | 10th percentile | `percentile(Flux, 10)` |
| `FluxP25` | 25th percentile | `percentile(Flux, 25)` |
| `FluxP75` | 75th percentile | `percentile(Flux, 75)` |
| `FluxP90` | 90th percentile | `percentile(Flux, 90)` |
| `FluxP95` | 95th percentile | `percentile(Flux, 95)` |
| `FluxP99` | 99th percentile | `percentile(Flux, 99)` |
| `FluxIqr` | robust middle spread | `P75 - P25` |
| `FluxAmplitude` | full observed flux range | `max(Flux) - min(Flux)` |
| `FluxPercentAmplitude95To5` | robust amplitude | `P95 - P05` |
| `FluxPercentAmplitude90To10` | more conservative robust amplitude | `P90 - P10` |
| `FluxSkewness` | asymmetry of flux distribution | `scipy.stats.skew(Flux)` |
| `FluxKurtosis` | peakedness / heavy-tail behavior | `scipy.stats.kurtosis(Flux)` |

## 4. Tail-Asymmetry Features

These features are intended to distinguish upward spikes from downward dips.

| Feature | Meaning | Method |
|---|---|---|
| `TailUpper` | strength of upper-brightness tail | `P95 - P50` |
| `TailLower` | strength of lower-brightness tail | `P50 - P05` |
| `TailAsymmetry` | positive = upper spikes dominate; negative = dips dominate | `TailUpper - TailLower` |
| `TailRatio` | ratio of upper-tail to lower-tail strength | `TailUpper / (TailLower + eps)` |

## 5. Time-Domain Variability Features

| Feature | Meaning | Method |
|---|---|---|
| `EtaVonNeumann` | smoothness / serial-correlation proxy | `sum(diff(Flux)^2) / ((N-1) * var(Flux))` |
| `MaxAbsSlope` | largest brightness change rate | `max(abs(diff(Flux) / diff(Time)))` |
| `MedianAbsSuccessiveDiff` | typical point-to-point brightness change | `median(abs(diff(Flux)))` |
| `FractionBeyond1Std` | fraction farther than 1 std from mean | `mean(abs(Flux - mean) > std)` |
| `FractionBeyond2Std` | fraction farther than 2 std from mean | `mean(abs(Flux - mean) > 2*std)` |

## 6. Lomb-Scargle Period Features

| Feature | Meaning | Method |
|---|---|---|
| `LsBestPeriod` | dominant period in days | `1 / frequency[argmax(power)]` |
| `LsBestFrequency` | dominant frequency | `frequency[argmax(power)]` |
| `LsMaxPower` | strength of dominant periodic signal | `max(power)` |
| `LsFalseAlarmProbability` | chance probability of dominant periodogram peak | `false_alarm_probability(maxPower)` |
| `LsPeriod2` | second strongest period | second-highest periodogram peak |
| `LsPeriod3` | third strongest period | third-highest periodogram peak |
| `LsPower2` | second strongest peak power | second-highest power |
| `LsPower3` | third strongest peak power | third-highest power |
| `LsPowerRatio21` | second/first power ratio | `Power2 / Power1` |
| `LsPowerRatio31` | third/first power ratio | `Power3 / Power1` |
| `LsPeriodRatio21` | second/first period ratio | `Period2 / Period1` |
| `LsPeriodRatio31` | third/first period ratio | `Period3 / Period1` |

## 7. Phase-Folded Morphology Features

| Feature | Meaning | Method |
|---|---|---|
| `PhaseCurveStd` | scatter of binned phase curve | `std(binnedFlux)` |
| `PhaseCurveRange` | amplitude of binned phase curve | `max(binnedFlux) - min(binnedFlux)` |
| `PhaseCurveSmoothness` | jaggedness/smoothness of folded curve | `std(diff(cyclicBinnedFlux))` |
| `PhasePeakPhase` | phase location of maximum binned flux | `phase[argmax(binnedFlux)]` |
| `PhaseTroughPhase` | phase location of minimum binned flux | `phase[argmin(binnedFlux)]` |
| `PhasePeakToTroughPhaseDelta` | cyclic phase distance from peak to trough | `min(abs(delta), 1 - abs(delta))` |

## 8. Extraction Status Features

| Feature | Meaning | Method / Source |
|---|---|---|
| `FeatureStatus` | extraction result: ok / failed / missing file / too few cadences | generated by extractor |
| `FeatureError` | error message if extraction failed | generated by extractor |


# Imports and Configuration

In [ ]:
from __future__ import annotations

import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import numpy as np
import pandas as pd

import lightkurve as lk
from astropy.io import fits
from astropy.timeseries import LombScargle
from scipy.stats import skew, kurtosis

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s:%(name)s:%(message)s",
)

Logger = logging.getLogger("FeatureExtractorNotebook")

# Update these paths for your local repo.
# InputMetadataPath = Path("../trend_detection/TESSAugmented_QC_trend.parquet")
# InputMetadataPath = Path("/data/projects/TESS-research/data_pipeline/TESSCache/TESSAugmented_QC.parquet")
# OutputFeaturePath = Path("/data/projects/TESS-research/feature_extraction/TESS_features.parquet")
InputMetadataPath = Path("/data/projects/TESS-research/detrend_tesscut/output/TESSAugmented_QC_tesscut_conditional_detrended.parquet")
OutputFeaturePath = Path("TESS_conditional_detrended_features.parquet")

MinPeriodDays = 0.05
MaxPeriodDays = 100.0
SamplesPerPeak = 10
MinCadences = 100
PhaseBinCount = 20
WorkerCount = 16
Eps = 1e-12

QualityScoreMap = {
    "clean": 0,
    "acceptable": 1,
    "caution": 2,
    "poor": 3,
    "missing": 4,
}

ProvenanceScoreMap = {
    "SPOC": 0,
    "QLP": 1,
    "TESSCut": 2,
}


# Utility Functions

In [ ]:
def SafeFloat(Value: Any) -> float:
    try:
        FloatValue = float(Value)
    except Exception:
        return np.nan
    return FloatValue if np.isfinite(FloatValue) else np.nan


def SafeBool(Value: Any) -> bool:
    if Value is None:
        return False
    if isinstance(Value, float) and pd.isna(Value):
        return False
    return bool(Value)


def SafeDivide(Numerator: float, Denominator: float, Eps: float = Eps) -> float:
    if not np.isfinite(Numerator) or not np.isfinite(Denominator) or abs(Denominator) < Eps:
        return np.nan
    return float(Numerator / Denominator)


def ResolvePath(PathValue: Any, MetadataPath: Path) -> Optional[Path]:
    if PathValue is None:
        return None
    if isinstance(PathValue, float) and pd.isna(PathValue):
        return None

    PathObj = Path(str(PathValue))
    if PathObj.is_absolute() and PathObj.exists():
        return PathObj
    if PathObj.exists():
        return PathObj

    CandidatePath = MetadataPath.parent / PathObj
    if CandidatePath.exists():
        return CandidatePath

    return PathObj


# Light Curve Loading

In [ ]:
def LoadLightCurve(LightCurvePath: Path) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Load a light curve while treating flux uncertainty as optional.

    Time and flux determine whether a cadence is scientifically usable.
    FluxErr is retained only when it is aligned with the loaded arrays and all
    retained uncertainty values are finite and strictly positive. Invalid or
    missing uncertainty therefore downgrades Lomb-Scargle from weighted to
    unweighted analysis instead of removing otherwise valid QLP cadences.
    """

    def FirstAvailableColumn(ColumnNames: list[str], CandidateNames: list[str]) -> Optional[str]:
        NameMap = {Name.upper(): Name for Name in ColumnNames}
        for CandidateName in CandidateNames:
            MatchedName = NameMap.get(CandidateName.upper())
            if MatchedName is not None:
                return MatchedName
        return None

    def ReadGenericFitsTable(FitsPath: Path) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
        with fits.open(str(FitsPath), memmap=False) as Hdul:
            TableData = None

            for Hdu in Hdul:
                Data = getattr(Hdu, "data", None)
                if Data is not None and getattr(Data, "dtype", None) is not None and Data.dtype.names:
                    TableData = Data
                    break

            if TableData is None:
                raise ValueError("No FITS binary table with named columns was found")

            ColumnNames = list(TableData.dtype.names)

            TimeColumn = FirstAvailableColumn(
                ColumnNames,
                ["TIME", "TMID", "BJD"],
            )
            FluxColumn = FirstAvailableColumn(
                ColumnNames,
                ["FLUX", "SAP_FLUX", "PDCSAP_FLUX", "KSPSAP_FLUX", "DET_FLUX", "NORM_FLUX"],
            )
            FluxErrColumn = FirstAvailableColumn(
                ColumnNames,
                ["FLUX_ERR", "SAP_FLUX_ERR", "PDCSAP_FLUX_ERR", "KSPSAP_FLUX_ERR", "ERR_FLUX"],
            )

            if TimeColumn is None or FluxColumn is None:
                raise ValueError(
                    f"Required time/flux columns are missing. Available columns: {ColumnNames}"
                )

            Time = np.asarray(TableData[TimeColumn], dtype=float).reshape(-1)
            Flux = np.asarray(TableData[FluxColumn], dtype=float).reshape(-1)
            FluxErr = None

            if FluxErrColumn is not None:
                FluxErr = np.asarray(TableData[FluxErrColumn], dtype=float).reshape(-1)

            return Time, Flux, FluxErr

    def ExtractFromLightkurve(LightCurve) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
        Time = np.asarray(
            getattr(LightCurve.time, "value", LightCurve.time),
            dtype=float,
        ).reshape(-1)

        Flux = np.asarray(
            getattr(LightCurve.flux, "value", LightCurve.flux),
            dtype=float,
        ).reshape(-1)

        FluxErr = None
        if hasattr(LightCurve, "flux_err") and LightCurve.flux_err is not None:
            FluxErr = np.asarray(
                getattr(LightCurve.flux_err, "value", LightCurve.flux_err),
                dtype=float,
            ).reshape(-1)

        return Time, Flux, FluxErr

    LightCurvePath = Path(LightCurvePath)

    if not LightCurvePath.exists():
        raise FileNotFoundError(str(LightCurvePath))

    Time = None
    Flux = None
    FluxErr = None
    ReadErrors = []

    # Attempt 1: standard lightkurve reader.
    try:
        LightCurve = lk.read(str(LightCurvePath))
        Time, Flux, FluxErr = ExtractFromLightkurve(LightCurve)
    except Exception as Exc:
        ReadErrors.append(f"generic lk.read: {Exc!r}")

    # Attempt 2: QLP-specific reader.
    if Time is None or Flux is None:
        try:
            LightCurve = lk.read(
                str(LightCurvePath),
                author="QLP",
                flux_column="kspsap_flux",
            )
            Time, Flux, FluxErr = ExtractFromLightkurve(LightCurve)
        except Exception as Exc:
            ReadErrors.append(f"QLP lk.read: {Exc!r}")

    # Attempt 3: direct FITS table parsing.
    if Time is None or Flux is None:
        try:
            Time, Flux, FluxErr = ReadGenericFitsTable(LightCurvePath)
        except Exception as Exc:
            ReadErrors.append(f"generic FITS parsing: {Exc!r}")
            raise ValueError(
                "Unable to read light curve. " + " | ".join(ReadErrors)
            ) from Exc

    if Time.shape != Flux.shape:
        raise ValueError(
            f"Time/Flux shape mismatch: Time={Time.shape}, Flux={Flux.shape}"
        )

    # Only Time and Flux define a valid cadence.
    CadenceMask = np.isfinite(Time) & np.isfinite(Flux)

    FilteredTime = Time[CadenceMask]
    FilteredFlux = Flux[CadenceMask]
    FilteredFluxErr = None

    # FluxErr is optional. It must never remove otherwise valid Time/Flux rows.
    if FluxErr is not None:
        if FluxErr.shape == Time.shape:
            CandidateFluxErr = FluxErr[CadenceMask]
            ValidFluxErr = np.isfinite(CandidateFluxErr) & (CandidateFluxErr > 0)

            if len(CandidateFluxErr) > 0 and np.all(ValidFluxErr):
                FilteredFluxErr = CandidateFluxErr
            else:
                Logger.debug(
                    "Ignoring unusable flux_err for %s: %d/%d retained values are finite and positive",
                    LightCurvePath,
                    int(np.sum(ValidFluxErr)),
                    len(CandidateFluxErr),
                )
        else:
            Logger.debug(
                "Ignoring flux_err shape mismatch for %s: flux_err=%s, time=%s",
                LightCurvePath,
                FluxErr.shape,
                Time.shape,
            )

    # Keep samples in chronological order.
    Order = np.argsort(FilteredTime)
    FilteredTime = FilteredTime[Order]
    FilteredFlux = FilteredFlux[Order]

    if FilteredFluxErr is not None:
        FilteredFluxErr = FilteredFluxErr[Order]

    return FilteredTime, FilteredFlux, FilteredFluxErr


# Identification and Metadata Features

In [ ]:
def ExtractIdentifierAndMetadataFeatures(Row: pd.Series) -> Dict[str, Any]:
    Result: Dict[str, Any] = {}

    ColumnsToPreserve = [
        "family", "VSXType", "VSXId", "Name", "ticId", "bestTicId",
        "ticDistanceArcmin", "lightCurvePath", "rawLightCurvePath",
        "trendFlag", "trendScore", "adfPValue",
    ]

    for ColumnName in ColumnsToPreserve:
        if ColumnName in Row.index:
            Result[ColumnName] = Row.get(ColumnName)

    QualityValue = Row.get("quality", Row.get("fitsQcStatus", Row.get("lightCurveQuality", np.nan)))
    ProvenanceValue = Row.get("provenance", Row.get("author", np.nan))

    QualityLabel = str(QualityValue) if not pd.isna(QualityValue) else "missing"
    ProvenanceLabel = str(ProvenanceValue) if not pd.isna(ProvenanceValue) else "missing"

    Result.update({
        "QualityLabel": QualityLabel,
        "QualityScore": QualityScoreMap.get(QualityLabel, np.nan),
        "Provenance": ProvenanceLabel,
        "ProvenanceScore": ProvenanceScoreMap.get(ProvenanceLabel, np.nan),
        "OriginalFluxMedian": SafeFloat(Row.get("fluxMedian", np.nan)),
        "OriginalFluxStd": SafeFloat(Row.get("fluxStd", np.nan)),
        "OriginalFluxSnr": SafeFloat(Row.get("fluxSnr", np.nan)),
        "LowSnr": SafeBool(Row.get("lowSNR", False)),
        "LowQualityLightCurve": SafeBool(Row.get("lowQualityLightCurve", False)),
    })
    return Result


# Data Coverage and Flux Distribution Features

In [ ]:
def ExtractBasicStatisticalFeatures(Time: np.ndarray, Flux: np.ndarray) -> Dict[str, float]:
    CadenceCount = len(Flux)
    P01, P05, P10, P25, P50, P75, P90, P95, P99 = np.percentile(
        Flux, [1, 5, 10, 25, 50, 75, 90, 95, 99]
    )
    FluxStd = float(np.std(Flux))

    return {
        "CadenceCount": float(CadenceCount),
        "TimeSpanDays": float(np.max(Time) - np.min(Time)) if CadenceCount > 1 else np.nan,
        "MedianCadenceDays": float(np.median(np.diff(Time))) if CadenceCount > 2 else np.nan,
        "FluxMean": float(np.mean(Flux)),
        "FluxStd": FluxStd,
        "FluxVariance": float(np.var(Flux)),
        "FluxMedian": float(P50),
        "FluxMad": float(np.median(np.abs(Flux - P50))),
        "FluxMin": float(np.min(Flux)),
        "FluxMax": float(np.max(Flux)),
        "FluxP01": float(P01),
        "FluxP05": float(P05),
        "FluxP10": float(P10),
        "FluxP25": float(P25),
        "FluxP75": float(P75),
        "FluxP90": float(P90),
        "FluxP95": float(P95),
        "FluxP99": float(P99),
        "FluxIqr": float(P75 - P25),
        "FluxAmplitude": float(np.max(Flux) - np.min(Flux)),
        "FluxPercentAmplitude95To5": float(P95 - P05),
        "FluxPercentAmplitude90To10": float(P90 - P10),
        "FluxSkewness": float(skew(Flux, bias=False)) if CadenceCount >= 3 and FluxStd > Eps else np.nan,
        "FluxKurtosis": float(kurtosis(Flux, bias=False)) if CadenceCount >= 4 and FluxStd > Eps else np.nan,
    }


# Tail-Asymmetry Features

In [ ]:
def ExtractTailAsymmetryFeatures(Flux: np.ndarray) -> Dict[str, float]:
    P05, P50, P95 = np.percentile(Flux, [5, 50, 95])

    TailUpper = float(P95 - P50)
    TailLower = float(P50 - P05)
    TailAsymmetry = float(TailUpper - TailLower)
    TailRatio = SafeDivide(TailUpper, TailLower + Eps)

    return {
        "TailUpper": TailUpper,
        "TailLower": TailLower,
        "TailAsymmetry": TailAsymmetry,
        "TailRatio": TailRatio,
    }


# Time-Domain Variability Features

In [ ]:
def ExtractVariabilityFeatures(Time: np.ndarray, Flux: np.ndarray) -> Dict[str, float]:
    CadenceCount = len(Flux)
    if CadenceCount < 3:
        return {
            "EtaVonNeumann": np.nan,
            "MaxAbsSlope": np.nan,
            "MedianAbsSuccessiveDiff": np.nan,
            "FractionBeyond1Std": np.nan,
            "FractionBeyond2Std": np.nan,
        }

    FluxMean = float(np.mean(Flux))
    FluxStd = float(np.std(Flux))
    FluxVariance = float(np.var(Flux))
    FluxDiff = np.diff(Flux)
    TimeDiff = np.diff(Time)

    ValidTimeDiff = np.isfinite(TimeDiff) & (np.abs(TimeDiff) > Eps)
    Slopes = FluxDiff[ValidTimeDiff] / TimeDiff[ValidTimeDiff] if np.any(ValidTimeDiff) else np.array([])

    Eta = np.sum(FluxDiff ** 2) / ((CadenceCount - 1) * FluxVariance) if FluxVariance > Eps else np.nan

    return {
        "EtaVonNeumann": float(Eta) if np.isfinite(Eta) else np.nan,
        "MaxAbsSlope": float(np.max(np.abs(Slopes))) if Slopes.size else np.nan,
        "MedianAbsSuccessiveDiff": float(np.median(np.abs(FluxDiff))),
        "FractionBeyond1Std": float(np.mean(np.abs(Flux - FluxMean) > FluxStd)) if FluxStd > Eps else np.nan,
        "FractionBeyond2Std": float(np.mean(np.abs(Flux - FluxMean) > 2 * FluxStd)) if FluxStd > Eps else np.nan,
    }


# Lomb-Scargle Period Features

In [ ]:
def ExtractLombScargleFeatures(Time: np.ndarray, Flux: np.ndarray, FluxErr: Optional[np.ndarray]) -> Dict[str, float]:
    Result = {
        "LsBestPeriod": np.nan,
        "LsBestFrequency": np.nan,
        "LsMaxPower": np.nan,
        "LsFalseAlarmProbability": np.nan,
        "LsPeriod2": np.nan,
        "LsPeriod3": np.nan,
        "LsPower2": np.nan,
        "LsPower3": np.nan,
        "LsPowerRatio21": np.nan,
        "LsPowerRatio31": np.nan,
        "LsPeriodRatio21": np.nan,
        "LsPeriodRatio31": np.nan,
    }

    if len(Flux) < MinCadences:
        return Result

    TimeSpan = float(np.max(Time) - np.min(Time))
    if not np.isfinite(TimeSpan) or TimeSpan <= 0:
        return Result

    MaxPeriod = min(MaxPeriodDays, 0.9 * TimeSpan)
    if MaxPeriod <= MinPeriodDays:
        return Result

    CenteredFlux = Flux - np.nanmedian(Flux)

    try:
        if FluxErr is not None and len(FluxErr) == len(Flux) and np.all(np.isfinite(FluxErr)):
            LombScargleModel = LombScargle(Time, CenteredFlux, dy=FluxErr)
        else:
            LombScargleModel = LombScargle(Time, CenteredFlux)

        Frequency, Power = LombScargleModel.autopower(
            minimum_frequency=1.0 / MaxPeriod,
            maximum_frequency=1.0 / MinPeriodDays,
            samples_per_peak=SamplesPerPeak,
        )

        FiniteMask = np.isfinite(Frequency) & np.isfinite(Power) & (Frequency > 0)
        Frequency = Frequency[FiniteMask]
        Power = Power[FiniteMask]
        if len(Power) == 0:
            return Result

        Order = np.argsort(Power)[::-1]
        BestFrequency = float(Frequency[Order[0]])
        BestPeriod = float(1.0 / BestFrequency)
        BestPower = float(Power[Order[0]])

        Result.update({
            "LsBestPeriod": BestPeriod,
            "LsBestFrequency": BestFrequency,
            "LsMaxPower": BestPower,
            "LsFalseAlarmProbability": float(LombScargleModel.false_alarm_probability(BestPower)),
        })

        if len(Order) > 1:
            Period2 = float(1.0 / Frequency[Order[1]])
            Power2 = float(Power[Order[1]])
            Result.update({
                "LsPeriod2": Period2,
                "LsPower2": Power2,
                "LsPowerRatio21": SafeDivide(Power2, BestPower),
                "LsPeriodRatio21": SafeDivide(Period2, BestPeriod),
            })

        if len(Order) > 2:
            Period3 = float(1.0 / Frequency[Order[2]])
            Power3 = float(Power[Order[2]])
            Result.update({
                "LsPeriod3": Period3,
                "LsPower3": Power3,
                "LsPowerRatio31": SafeDivide(Power3, BestPower),
                "LsPeriodRatio31": SafeDivide(Period3, BestPeriod),
            })

    except Exception as Exc:
        Logger.debug("Lomb-Scargle failed: %s", Exc)

    return Result


# Phase-Folded Morphology Features

In [ ]:
def ExtractPhaseFeatures(Time: np.ndarray, Flux: np.ndarray, Period: float) -> Dict[str, float]:
    Result = {
        "PhaseCurveStd": np.nan,
        "PhaseCurveRange": np.nan,
        "PhaseCurveSmoothness": np.nan,
        "PhasePeakPhase": np.nan,
        "PhaseTroughPhase": np.nan,
        "PhasePeakToTroughPhaseDelta": np.nan,
    }

    if not np.isfinite(Period) or Period <= 0 or len(Flux) < MinCadences:
        return Result

    try:
        Phase = (Time % Period) / Period
        Order = np.argsort(Phase)
        Phase = Phase[Order]
        Flux = Flux[Order]

        BinEdges = np.linspace(0.0, 1.0, PhaseBinCount + 1)
        BinIndex = np.digitize(Phase, BinEdges) - 1

        BinnedPhase = []
        BinnedFlux = []

        for BinNumber in range(PhaseBinCount):
            BinMask = BinIndex == BinNumber
            if np.any(BinMask):
                BinnedPhase.append(float(np.median(Phase[BinMask])))
                BinnedFlux.append(float(np.median(Flux[BinMask])))

        BinnedPhase = np.asarray(BinnedPhase, dtype=float)
        BinnedFlux = np.asarray(BinnedFlux, dtype=float)
        if len(BinnedFlux) < 5:
            return Result

        PeakIndex = int(np.argmax(BinnedFlux))
        TroughIndex = int(np.argmin(BinnedFlux))
        PeakPhase = float(BinnedPhase[PeakIndex])
        TroughPhase = float(BinnedPhase[TroughIndex])

        RawDelta = abs(PeakPhase - TroughPhase)
        CyclicDelta = min(RawDelta, 1.0 - RawDelta)
        CyclicDiff = np.diff(np.r_[BinnedFlux, BinnedFlux[0]])

        Result.update({
            "PhaseCurveStd": float(np.std(BinnedFlux)),
            "PhaseCurveRange": float(np.max(BinnedFlux) - np.min(BinnedFlux)),
            "PhaseCurveSmoothness": float(np.std(CyclicDiff)),
            "PhasePeakPhase": PeakPhase,
            "PhaseTroughPhase": TroughPhase,
            "PhasePeakToTroughPhaseDelta": float(CyclicDelta),
        })

    except Exception as Exc:
        Logger.debug("Phase feature extraction failed: %s", Exc)

    return Result


# Per-Star Feature Extraction

In [ ]:
def ExtractFeaturesForRow(Row: pd.Series, MetadataPath: Path) -> Dict[str, Any]:
    fitsBasePath = Path("/data/projects/TESS-research/data_pipeline")
    Result: Dict[str, Any] = {
        "FeatureStatus": "unknown",
        "FeatureError": None,
    }

    Result.update(ExtractIdentifierAndMetadataFeatures(Row))
    # LightCurvePath = ResolvePath(Row.get("lightCurvePath"), MetadataPath)
    LightCurvePath = Path(fitsBasePath / Row.get("lightCurvePath"))

    if LightCurvePath is None:
        Result["FeatureStatus"] = "missing_lightcurve_path"
        Result["FeatureError"] = "No lightCurvePath"
        return Result

    if not LightCurvePath.exists():
        Result["FeatureStatus"] = "missing_lightcurve_file"
        Result["FeatureError"] = str(LightCurvePath)
        return Result

    try:
        Time, Flux, FluxErr = LoadLightCurve(LightCurvePath)

        if len(Flux) < MinCadences:
            Result["FeatureStatus"] = "too_few_cadences"
            Result["FeatureError"] = f"Only {len(Flux)} finite cadences"
            Result["CadenceCount"] = float(len(Flux))
            return Result

        Result.update(ExtractBasicStatisticalFeatures(Time, Flux))
        Result.update(ExtractTailAsymmetryFeatures(Flux))
        Result.update(ExtractVariabilityFeatures(Time, Flux))

        LombScargleFeatures = ExtractLombScargleFeatures(Time, Flux, FluxErr)
        Result.update(LombScargleFeatures)

        BestPeriod = LombScargleFeatures.get("LsBestPeriod", np.nan)
        Result.update(ExtractPhaseFeatures(Time, Flux, BestPeriod))

        Result["FeatureStatus"] = "ok"

    except Exception as Exc:
        Result["FeatureStatus"] = "failed"
        Result["FeatureError"] = repr(Exc)

    return Result


# Dataset-Level Feature Extraction with Parallel Processing

In [ ]:
def FinalizeFeatureDf(FeatureDf: pd.DataFrame) -> pd.DataFrame:
    if "_InputOrder" in FeatureDf.columns:
        FeatureDf = FeatureDf.sort_values("_InputOrder").drop(columns=["_InputOrder"]).reset_index(drop=True)
    return FeatureDf


def ExtractFeaturesSerial(MetadataDf: pd.DataFrame, MetadataPath: Path) -> pd.DataFrame:
    ResultRows = []
    for Position, (_, Row) in enumerate(MetadataDf.iterrows()):
        if Position % 100 == 0:
            Logger.info("Processing %s/%s", Position, len(MetadataDf))
        FeatureRow = ExtractFeaturesForRow(Row, MetadataPath)
        FeatureRow["_InputOrder"] = Position
        ResultRows.append(FeatureRow)
    return FinalizeFeatureDf(pd.DataFrame(ResultRows))


def ExtractFeaturesParallel(MetadataDf: pd.DataFrame, MetadataPath: Path, WorkerCount: int = WorkerCount) -> pd.DataFrame:
    ResultRows = []
    TotalRows = len(MetadataDf)
    Logger.info("Starting parallel feature extraction with WorkerCount=%s", WorkerCount)

    with ThreadPoolExecutor(max_workers=WorkerCount) as Executor:
        FutureMap = {}
        for Position, (_, Row) in enumerate(MetadataDf.iterrows()):
            Future = Executor.submit(ExtractFeaturesForRow, Row, MetadataPath)
            FutureMap[Future] = Position

        CompletedCount = 0
        for Future in as_completed(FutureMap):
            Position = FutureMap[Future]
            try:
                FeatureRow = Future.result()
            except Exception as Exc:
                FeatureRow = {"FeatureStatus": "failed", "FeatureError": repr(Exc)}

            FeatureRow["_InputOrder"] = Position
            ResultRows.append(FeatureRow)
            CompletedCount += 1

            if CompletedCount % 100 == 0 or CompletedCount == TotalRows:
                Logger.info("Completed %s/%s", CompletedCount, TotalRows)

    return FinalizeFeatureDf(pd.DataFrame(ResultRows))


def ExtractFeatures(MetadataDf: pd.DataFrame, MetadataPath: Path, WorkerCount: int = WorkerCount) -> pd.DataFrame:
    if WorkerCount <= 1:
        return ExtractFeaturesSerial(MetadataDf, MetadataPath)
    return ExtractFeaturesParallel(MetadataDf, MetadataPath, WorkerCount=WorkerCount)


# Execute Feature Extraction

In [ ]:
MetadataDf = pd.read_parquet(InputMetadataPath)

FeatureDf = ExtractFeatures(
    MetadataDf=MetadataDf,
    MetadataPath=InputMetadataPath,
    WorkerCount=WorkerCount,
)

OutputFeaturePath.parent.mkdir(parents=True, exist_ok=True)
FeatureDf.to_parquet(OutputFeaturePath, index=False)

print(f"Saved {len(FeatureDf)} rows and {len(FeatureDf.columns)} columns to {OutputFeaturePath}")
FeatureDf["FeatureStatus"].value_counts(dropna=False)


# Quick Sanity Checks

In [ ]:
display(FeatureDf.head())

NumericColumns = FeatureDf.select_dtypes(include=[np.number]).columns
MissingRate = FeatureDf[NumericColumns].isna().mean().sort_values(ascending=False)
display(MissingRate.head(30))


In [ ]:
if "family" in FeatureDf.columns:
    display(pd.crosstab(FeatureDf["family"], FeatureDf["FeatureStatus"], dropna=False))
